In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
# -*- coding: utf-8 -*-
from __future__ import annotations
import re
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import pandas as pd

# Métricas tal como aparecen en la columna "metric" de cv_summary_full.csv
METRIC_KEYS = [
    "acc",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "precision_weighted",
    "recall_weighted",
    "f1_weighted",
]

RUN_PREFIX = "Run"                 # prefijo de las carpetas de experimento
CV_FILENAME = "cv_summary_full.csv"
OUT_CSV_NAME = "external_test_summary.csv"


def _extract_n_syn_from_name(name: str) -> Optional[int]:
    """
    Extrae el último número de un nombre de carpeta, p.ej.:
      'Run 0' -> 0
      'Run_500' -> 500
      'Run-1000' -> 1000
    """
    m = re.findall(r"(\d+)", name)
    if not m:
        return None
    return int(m[-1])


def _find_run_dirs(root: Path, prefix: str = RUN_PREFIX) -> List[Path]:
    """
    Devuelve todas las subcarpetas cuyo nombre empieza con `prefix`
    (por ejemplo: 'Run', 'Run ', 'Run_').
    """
    runs = []
    for p in root.iterdir():
        if p.is_dir() and p.name.startswith(prefix):
            runs.append(p)

    # Ordenamos por n_syn si lo podemos extraer
    runs_sorted = sorted(
        runs,
        key=lambda d: (
            _extract_n_syn_from_name(d.name)
            if _extract_n_syn_from_name(d.name) is not None else 1e9
        )
    )
    return runs_sorted


def _extract_external_test_metrics(df: pd.DataFrame) -> Dict[str, float]:
    """
    Recibe un cv_summary_full en formato:
        split | metric | mean | std | k
    Filtra split == 'test' (test externo) y devuelve
    { 'acc_mean': ..., 'acc_std': ..., ... }.
    """
    if "split" not in df.columns or "metric" not in df.columns:
        raise ValueError("Se esperaban columnas 'split' y 'metric'.")

    # nos quedamos con las filas de test externo
    mask = df["split"].astype(str).str.lower() == "test"
    df_test = df[mask].copy()
    if df_test.empty:
        raise ValueError("No se encontraron filas con split == 'test'.")

    df_test = df_test.set_index("metric")

    out: Dict[str, float] = {}
    for m in METRIC_KEYS:
        if m not in df_test.index:
            continue
        row = df_test.loc[m]
        out[f"{m}_mean"] = float(row["mean"])
        out[f"{m}_std"] = float(row["std"])
    return out


def build_external_test_summary(
    root: str | Path,
    run_prefix: str = RUN_PREFIX,
    cv_filename: str = CV_FILENAME,
    out_csv_name: str = OUT_CSV_NAME,
) -> pd.DataFrame:
    """
    Recorre carpetas tipo 'Run 0', 'Run 100', etc. dentro de `root`,
    lee `cv_summary_full.csv` en cada una y arma una tabla consolidada
    SOLO con métricas de test externo (split == 'test').

    Devuelve un DataFrame con columnas:
      run_name, n_syn,
      test_acc_mean, test_acc_std, test_precision_macro_mean, ...
    y lo guarda en `root/out_csv_name`.
    """
    root = Path(root)
    run_dirs = _find_run_dirs(root, prefix=run_prefix)
    if not run_dirs:
        raise SystemExit(f"No se encontraron carpetas que empiecen con '{run_prefix}' en {root}")

    rows = []
    for rdir in run_dirs:
        cv_path = rdir / cv_filename
        if not cv_path.exists():
            print(f"[WARN] No se encontró {cv_path}, se omite '{rdir.name}'.")
            continue

        df = pd.read_csv(cv_path)
        try:
            metrics = _extract_external_test_metrics(df)
        except Exception as e:
            print(f"[WARN] {cv_path}: no pude extraer test externo: {e}")
            continue

        row: Dict[str, float | int | str] = {}
        row["run_name"] = rdir.name
        n_syn = _extract_n_syn_from_name(rdir.name)
        row["n_syn"] = n_syn if n_syn is not None else -1

        # renombramos con prefijo 'test_'
        for m in METRIC_KEYS:
            row[f"test_{m}_mean"] = metrics.get(f"{m}_mean", np.nan)
            row[f"test_{m}_std"] = metrics.get(f"{m}_std", np.nan)

        rows.append(row)

    if not rows:
        raise SystemExit("No se pudo construir ninguna fila; revisa tus cv_summary_full.csv.")

    out_df = pd.DataFrame(rows).sort_values("n_syn").reset_index(drop=True)
    out_path = root / out_csv_name
    out_df.to_csv(out_path, index=False)
    print(f"[OK] Resumen de TEST EXTERNO guardado en: {out_path}")
    print(out_df)
    return out_df


# ----------------------------
# Ejemplo de uso
# ----------------------------
if __name__ == "__main__":
    ROOT = "/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfold"  # <--- cambia a tu raíz
    _ = build_external_test_summary(
        ROOT,
        run_prefix="Run",
        cv_filename="cv_summary_full.csv",
        out_csv_name="external_test_summary.csv",
    )


[WARN] /content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfold/Run 0/cv_summary_full.csv: no pude extraer test externo: No se encontraron filas con split == 'test'.
[WARN] /content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfold/Run 100/cv_summary_full.csv: no pude extraer test externo: No se encontraron filas con split == 'test'.
[WARN] /content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfold/Run 200/cv_summary_full.csv: no pude extraer test externo: No se encontraron filas con split == 'test'.
[WARN] /content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfold/Run 300/cv_summary_full.csv: no pude extraer test externo: No se encontraron filas con split == 'test'.
[WARN] /content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfold/Run 400/cv_summary_full.csv: no pude extraer test externo: No se encontraron filas con split == 'test'.
[WARN] /content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfold/Run 500/cv_summary_full.csv: no pude extraer test externo: No se encontraron filas con split == 'test'.


SystemExit: No se pudo construir ninguna fila; revisa tus cv_summary_full.csv.

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [8]:
from IPython.display import Image, display
import os

plots_dir = Path(ROOT) / "INTTEST_plots"

print(f"Displaying plots from: {plots_dir}")

# List all image files in the directory
image_files = [f for f in os.listdir(plots_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

if not image_files:
    print("No image files found in the specified directory.")
else:
    # Display each image
    for image_file in image_files:
        image_path = plots_dir / image_file
        print(f"\nDisplaying: {image_file}")
        display(Image(filename=image_path))

Displaying plots from: /content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldnuevo/INTTEST_plots


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/TESIS/StyleGan2/CNNResnet50kfoldnuevo/INTTEST_plots'